# Notebook 06 — Pipeline tabulaire avec variables catégorielles

## Objectif
Intégrer les colonnes catégorielles (`gene`, `cohort`, `rater_id`) avec `skrub.tabular_pipeline`. Comparer ce modèle enrichi au meilleur modèle numérique sur les mêmes folds GroupKFold.

## Fichiers utilisés
- `data/X_train.csv` — variables explicatives
- `data/y_train.csv` — variable cible

## Résultat attendu
- RMSE GroupKFold du pipeline tabulaire
- Comparaison avec HGBR numérique
- Rapport Skore Hub (clé `05_tabular`)

## Notebook suivant
`07_feature_engineering.ipynb` — création de nouvelles variables.

## 1. Imports

In [9]:
import json
import os
from pathlib import Path

import pandas as pd
import skore
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from skrub import tabular_pipeline

## 2. Connexion à Skore Hub

In [10]:
config = json.loads(Path("../.skore").read_text())
os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
skore.login()
hub_project = skore.Project(
    "bobathon-esilv",
    mode="hub",
    workspace=config["workspace"],
)
print(f"Connecté au Hub : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                                  Already logged in.                                  │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté au Hub : workspace=BERNIE MADOFF


## 3. Chargement des données

In [11]:
DATA_DIR = Path("../data")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")

visits = X_train.merge(
    y_train,
    on="Index",
    how="inner",
    validate="one_to_one",
)

assert len(visits) == len(X_train)
assert visits["target"].notna().all()

print(f"visits : {visits.shape} — OK")

visits : (44590, 14) — OK


## 4. Préparation des features

On supprime `Index`, `patient_id` et `target`. On conserve toutes les autres colonnes — numériques et catégorielles — pour les passer au pipeline tabulaire.

`rater_id` est inclus à titre de test : il peut capturer un effet réel de l'évaluateur, mais aussi introduire un biais si certains évaluateurs ne voient qu'un sous-groupe de patients.

In [12]:
X_full = visits.drop(columns=["Index", "patient_id", "target"])
y = visits["target"]
groups = visits["patient_id"]

print(f"X_full : {X_full.shape}")
print("Colonnes :", list(X_full.columns))
print("\nTypes :")
print(X_full.dtypes)

X_full : (44590, 11)
Colonnes : ['cohort', 'sexM', 'gene', 'age_at_diagnosis', 'age', 'ledd', 'time_since_intake_on', 'time_since_intake_off', 'rater_id', 'on', 'off']

Types :
cohort                       str
sexM                       int64
gene                         str
age_at_diagnosis         float64
age                      float64
ledd                     float64
time_since_intake_on     float64
time_since_intake_off    float64
rater_id                     str
on                       float64
off                      float64
dtype: object


## 5. Folds GroupKFold (identiques aux notebooks précédents)

In [13]:
cv_splits = list(GroupKFold(n_splits=5).split(X_full, y, groups=groups))
print(f"{len(cv_splits)} folds GroupKFold — {groups.nunique()} patients")

5 folds GroupKFold — 5576 patients


## 6. Pipeline tabulaire skrub

`skrub.tabular_pipeline("regressor")` construit automatiquement un pipeline adapté aux colonnes mixtes : encodage des catégorielles, imputation des numériques, et un `HistGradientBoostingRegressor` par défaut.

In [14]:
tabular_model = tabular_pipeline("regressor")
print(tabular_model)

Pipeline(steps=[('tablevectorizer',
                 TableVectorizer(low_cardinality=ToCategorical())),
                ('histgradientboostingregressor',
                 HistGradientBoostingRegressor())])


## 7. Référence HGBR numérique (mêmes folds)

In [15]:
feature_cols_num = [
    "sexM", "age_at_diagnosis", "age", "ledd",
    "time_since_intake_on", "time_since_intake_off", "on", "off",
]
X_num = visits[feature_cols_num]

hgbr = HistGradientBoostingRegressor(random_state=0)
# Les folds sont recalculés sur X_num (même groupes, même splits)
cv_splits_num = list(GroupKFold(n_splits=5).split(X_num, y, groups=groups))

report_hgbr = skore.evaluate(hgbr, X_num, y, splitter=cv_splits_num)
rmse_hgbr = float(report_hgbr.metrics.rmse().values.flatten().mean())
print(f"RMSE HGBR numérique (GroupKFold, moyenne) : {rmse_hgbr:.4f}")

Output()

RMSE HGBR numérique (GroupKFold, moyenne) : 3.9533


## 8. Évaluation du pipeline tabulaire

In [16]:
report_tabular = skore.evaluate(tabular_model, X_full, y, splitter=cv_splits)
rmse_tabular = float(report_tabular.metrics.rmse().values.flatten().mean())
print(f"RMSE tabular_pipeline (GroupKFold, moyenne) : {rmse_tabular:.4f}")

Output()

RMSE tabular_pipeline (GroupKFold, moyenne) : 3.8300


## 9. Tableau comparatif

In [17]:
hgbr_vals    = report_hgbr.metrics.rmse().iloc[:, 0].values
tabular_vals = report_tabular.metrics.rmse().iloc[:, 0].values

n_folds = len(hgbr_vals)
comparison = pd.DataFrame({
    "HGBR numérique": hgbr_vals,
    "Tabular pipeline": tabular_vals,
}, index=[f"Fold {i+1}" for i in range(n_folds)])
comparison.loc["Moyenne"]    = comparison.mean()
comparison.loc["Écart-type"] = comparison.iloc[:n_folds].std()

print("RMSE par fold :")
display(comparison.round(4))

RMSE par fold :


,HGBR numérique,Tabular pipeline
Fold 1,7.752,7.5262
Moyenne,7.752,7.5262
Écart-type,NaN,NaN


## 10. Envoi du rapport Skore Hub

In [18]:
hub_project.put("05_tabular", report_tabular)
print("Rapport envoyé à Skore Hub sous la clé : 05_tabular")

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/cross-validations/61307

Rapport envoyé à Skore Hub sous la clé : 05_tabular


## 11. Conclusion

**Points clés :**
- `skrub.tabular_pipeline` encode automatiquement les colonnes catégorielles (`gene`, `cohort`, `rater_id`) sans préparation manuelle.
- `Index` et `patient_id` sont supprimés avant de passer au modèle.
- `rater_id` peut améliorer le score ou introduire un biais : l'analyser dans le notebook d'erreurs.
- Si le tabular pipeline bat HGBR numérique, les catégorielles apportent de l'information.

**Prochaine étape** : `07_feature_engineering.ipynb` — création de nouvelles variables calculées.